In [5]:
import pandas as pd

filtered_metadata = pd.read_csv("../../r232/filtered_representative_genomes.tsv", sep="\t")

taxids = pd.read_csv("../../r232/taxdump_r232.tsv", sep="\t")


In [6]:
filtered_metadata.head(10)

,accession,ambiguous_bases,checkm2_completeness,checkm2_contamination,checkm2_model,checkm_completeness,checkm_contamination,checkm_marker_count,checkm_marker_lineage,checkm_marker_set_count,...,ssu_silva_blast_bitscore,ssu_silva_blast_evalue,ssu_silva_blast_perc_identity,ssu_silva_blast_subject_id,ssu_silva_taxonomy,total_gap_length,trna_aa_count,trna_count,trna_selenocysteine_count,domain
0,GB_GCA_018646675.1,0,94.42,1.36,General,91.22,0.89,182,k__Bacteria (UID3187),112,...,none,none,none,none,none,0,19,35,0,Bacteria
1,GB_GCA_017517145.1,0,80.39,1.18,Specific,87.36,3.52,263,o__Clostridiales (UID1212),149,...,none,none,none,none,none,0,14,33,0,Bacteria
2,GB_GCA_045656825.1,0,85.00,1.06,General,79.19,1.14,263,o__Clostridiales (UID1212),149,...,none,none,none,none,none,0,13,23,0,Bacteria
3,GB_GCA_023268935.1,0,89.33,2.22,Specific,91.80,2.91,693,c__Gammaproteobacteria (UID4761),297,...,none,none,none,none,none,0,15,31,0,Bacteria
4,RS_GCF_031457235.1,0,100.00,1.36,Specific,99.61,0.70,427,o__Burkholderiales (UID4000),214,...,2745,0,99.212,AJ585992.1.1527,Bacteria;Pseudomonadota;Gammaproteobacteria;Bu...,0,19,45,0,Bacteria
5,GB_GCA_030430655.1,0,88.92,1.18,Specific,91.05,0.68,230,k__Bacteria (UID2982),148,...,none,none,none,none,none,0,18,32,0,Bacteria
6,RS_GCF_964511145.1,105,100.00,0.18,Specific,99.66,0.59,548,f__Flavobacteriaceae (UID2845),298,...,2673,0,100,AB681061.1.1447,Bacteria;Bacteroidota;Bacteroidia;Flavobacteri...,0,18,42,0,Bacteria
7,GB_GCA_963842765.1,0,64.21,0.31,Specific,72.07,0.00,104,k__Bacteria (UID203),58,...,2760,0,100,HQ190966.1.1494,Bacteria;Pseudomonadota;Gammaproteobacteria;St...,0,18,30,0,Bacteria
8,GB_GCA_964592055.1,0,90.75,1.77,Specific,95.20,2.23,270,k__Bacteria (UID2570),179,...,none,none,none,none,none,11910,17,33,0,Bacteria
9,GB_GCA_934196075.1,0,88.54,0.15,Specific,93.65,1.31,295,p__Firmicutes (UID1022),158,...,none,none,none,none,none,0,18,48,0,Bacteria


In [8]:
taxids.head(20)

,taxID,name,rank
0,1,root,no rank
1,2,d__Bacteria,superkingdom
2,3,p__Pseudomonadota,phylum
3,4,c__Gammaproteobacteria,class
4,5,o__Enterobacterales,order
5,6,f__Enterobacteriaceae,family
6,7,g__Escherichia,genus
7,8,s__Escherichia coli,species
8,9,RS_GCF_051534235.1,subspecies
9,10,RS_GCF_046376025.1,subspecies


In [12]:
# Prepare the right table
right_table = taxids[['taxID', 'name']].rename(columns={'taxID': 'gtdb_taxid'})

# Inner join
meta = filtered_metadata.merge(
    right_table,
    left_on='accession',    # join key from meta
    right_on='name',        # join key from right_table
    how='inner'
)

In [14]:
# Write the table

meta.to_csv("../../r232/filtered_metadata_with_taxids.tsv", sep="\t", index=False)

## Genomes

Original struo2 downloads just the genomes needed. I already have all the genomes downloaded, so I just need to append the metadata table with the file locations.


In [20]:
# Genome dirs

genomes_dir = "../../r232/genome_fasta/gtdb_genomes_reps_r232"

In [23]:
# map the genomes dir helper function 

from pathlib import Path
import pandas as pd

def build_genome_table(genomes_dir: str, pattern: str = "*_genomic.fna.gz") -> pd.DataFrame:
    """
    Walk genomes_dir/database/{XXX}/{XXX}/{XXX}/... and build a table of
    accession numbers (with GB_/RS_ prefix) and their absolute file paths.

    Parameters
    ----------
    genomes_dir : str
        Root directory containing the "database" subfolder structure.
    pattern : str
        Glob pattern to match genome files (default: "*_genomic.fna.gz").

    Returns
    -------
    pd.DataFrame with columns: accession, path
    """
    root = Path(genomes_dir).resolve() / "database"

    if not root.exists():
        raise FileNotFoundError(f"Database directory not found: {root}")

    prefix_map = {"GCA": "GB_", "GCF": "RS_"}

    records = []
    for fpath in root.rglob(pattern):
        stem = fpath.name
        # e.g. GCA_018646675.1_genomic.fna.gz -> accession = GCA_018646675.1
        base = stem.split("_genomic")[0]
        parts = base.split("_")
        bare_accession = "_".join(parts[:2]) if len(parts) >= 2 else base

        db_code = bare_accession.split("_")[0]  # GCA or GCF
        prefix = prefix_map.get(db_code, "")
        accession = f"{prefix}{bare_accession}"

        records.append({
            "accession": accession,
            "path": str(fpath.resolve()),
        })

    df = pd.DataFrame(records, columns=["accession", "path"])
    return df.sort_values("accession").reset_index(drop=True)


# Example usage:
# df = build_genome_table("/path/to/genomes_dir")
# print(df.head())


In [ ]:
genomes_df = build_genome_table(genomes_dir)
genomes_df

,accession,path
0,GB_GCA_000008085.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
1,GB_GCA_000008885.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
2,GB_GCA_000010565.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
3,GB_GCA_000012145.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
4,GB_GCA_000013625.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
5,GB_GCA_000016765.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
6,GB_GCA_000017645.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
7,GB_GCA_000018565.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
8,GB_GCA_000023185.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
9,GB_GCA_000023405.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...


In [25]:
metadata_accessions = set(meta['accession'])
genome_accessions = set(genomes_df['accession'])

# Find accessions in metadata but not in genomes
missing_in_genomes = metadata_accessions - genome_accessions
print(f"Accessions in metadata but not in genomes: {len(missing_in_genomes)}")

# Find accessions in genomes but not in metadata
missing_in_metadata = genome_accessions - metadata_accessions
print(f"Accessions in genomes but not in metadata: {len(missing_in_metadata)}")


Accessions in metadata but not in genomes: 0
Accessions in genomes but not in metadata: 9611


This makes sense all the metadata accessions were found in the genomes... there are more genomes because the metadata is filtered. 

In [26]:
# append file paths to the metadata table

meta = meta.merge(genomes_df, on="accession", how="left")

In [27]:
meta

,accession,ambiguous_bases,checkm2_completeness,checkm2_contamination,checkm2_model,checkm_completeness,checkm_contamination,checkm_marker_count,checkm_marker_lineage,checkm_marker_set_count,...,ssu_silva_blast_subject_id,ssu_silva_taxonomy,total_gap_length,trna_aa_count,trna_count,trna_selenocysteine_count,domain,gtdb_taxid,name,path
0,GB_GCA_018646675.1,0,94.42,1.36,General,91.22,0.89,182,k__Bacteria (UID3187),112,...,none,none,0,19,35,0,Bacteria,650078,GB_GCA_018646675.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
1,GB_GCA_017517145.1,0,80.39,1.18,Specific,87.36,3.52,263,o__Clostridiales (UID1212),149,...,none,none,0,14,33,0,Bacteria,873054,GB_GCA_017517145.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
2,GB_GCA_045656825.1,0,85.00,1.06,General,79.19,1.14,263,o__Clostridiales (UID1212),149,...,none,none,0,13,23,0,Bacteria,1080761,GB_GCA_045656825.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
3,GB_GCA_023268935.1,0,89.33,2.22,Specific,91.80,2.91,693,c__Gammaproteobacteria (UID4761),297,...,none,none,0,15,31,0,Bacteria,585531,GB_GCA_023268935.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
4,RS_GCF_031457235.1,0,100.00,1.36,Specific,99.61,0.70,427,o__Burkholderiales (UID4000),214,...,AJ585992.1.1527,Bacteria;Pseudomonadota;Gammaproteobacteria;Bu...,0,19,45,0,Bacteria,860080,RS_GCF_031457235.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190307,GB_GCA_029850515.1,0,99.55,0.13,Specific,99.51,0.97,145,k__Archaea (UID2),103,...,none,none,0,18,44,0,Archaea,1133985,GB_GCA_029850515.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
190308,GB_GCA_046732455.1,0,79.09,3.35,Specific,85.52,3.40,145,k__Archaea (UID2),103,...,none,none,0,18,36,0,Archaea,1142607,GB_GCA_046732455.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
190309,GB_GCA_964588555.1,0,92.48,1.72,General,91.82,2.80,149,k__Archaea (UID2),107,...,HQ395738.1.1465,Archaea;Micrarchaeota;Micrarchaeia;Micrarchaea...,0,13,16,0,Archaea,1146943,GB_GCA_964588555.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...
190310,GB_GCA_041305945.1,0,63.56,2.43,General,66.00,2.34,149,k__Archaea (UID2),107,...,none,none,0,15,29,2,Archaea,1125561,GB_GCA_041305945.1,/mnt/Projects/Sam/pipelines/Struo2/r232/genome...


In [28]:
#save table with file paths

meta.to_csv("../../r232/filtered_metadata_with_taxids_and_paths.tsv", sep="\t", index=False)